# RadFM Backbone Verification

**Model**: RadFM (`chaoyi-wu/RadFM`) — Radiology Foundation Model  
**Arch**: 3D ViT image encoder + Perceiver resampler + LLaMA-7B decoder  
**Embed dim**: Per-image token set: `32 × d` (32 Perceiver tokens, dim d=4096 or 5120)  
           For a single global vector, mean-pool the 32 tokens → `[B, d]`  
**Input**: 2D CXR padded to 3D `(C=3, H, W, D=4)` — depth repeated  
**Access**: HuggingFace `chaoyi-wu/RadFM` (public)  
**Zero-shot / VQA**: Yes — full LLM generation  
**Hardware**: Requires GPU with ≥40GB VRAM (A100/H100) — CPU inference is impractical

Paper: [Towards Generalist Foundation Model for Radiology by Leveraging Web-scale 2D&3D Medical Data (Nature Communications, 2025)](https://www.nature.com/articles/s41467-025-62385-7)

---

> **Scope**: RadFM is primarily a **generative VQA model**, not a classification
> encoder. This notebook shows how to extract visual-only embeddings from the
> image encoder (ViT + Perceiver) without running the full LLaMA decoder.
> For linear-probe / kNN evaluation, the 32-token mean-pooled Perceiver output
> is a practical global descriptor.
>
> For full report generation / VQA, use the official RadFM inference pipeline instead.

### Prerequisites

```bash
git clone https://github.com/chaoyi-wu/RadFM /tmp/RadFM
cd /tmp/RadFM && pip install -r requirements.txt
# Download pytorch_model.bin from: https://huggingface.co/chaoyi-wu/RadFM
# Place at: /tmp/RadFM/Model/RadFM/pytorch_model.bin
```

In [ ]:
%pip install -q torch torchvision transformers einops Pillow

import subprocess, os, sys

RADFM_DIR = "/tmp/RadFM"
if not os.path.exists(RADFM_DIR):
    subprocess.run(["git", "clone", "https://github.com/chaoyi-wu/RadFM",
                    RADFM_DIR], check=True)

# Add the src directory to path
SRC_DIR = os.path.join(RADFM_DIR, "src", "model")
if os.path.exists(SRC_DIR):
    sys.path.insert(0, SRC_DIR)
sys.path.insert(0, RADFM_DIR)
print(os.listdir(RADFM_DIR))

In [ ]:
import os

# Control which GPU(s) are visible to this notebook.
# Set CUDA_VISIBLE_DEVICES in your shell before launching Jupyter to override,
# e.g.:  CUDA_VISIBLE_DEVICES=2 jupyter lab
# Or change the default value below.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
print(f"CUDA_VISIBLE_DEVICES = {os.environ['CUDA_VISIBLE_DEVICES']}")

## 1. Download model weights

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download
import os

# Option A: snapshot download (gets all files ~30GB)
# model_dir = snapshot_download("chaoyi-wu/RadFM", local_dir="/tmp/RadFM/Model/RadFM")

# Option B: single file download
CKPT_PATH = "/tmp/RadFM/Model/RadFM/pytorch_model.bin"
if not os.path.exists(CKPT_PATH):
    print("Downloading pytorch_model.bin (~26GB) ...")
    CKPT_PATH = hf_hub_download(
        repo_id="chaoyi-wu/RadFM",
        filename="pytorch_model.bin",
        local_dir="/tmp/RadFM/Model/RadFM",
    )

print("Checkpoint:", CKPT_PATH)
print("Size:", os.path.getsize(CKPT_PATH) / 1e9, "GB")

## 2. Load the full RadFM model

In [ ]:
import torch

# RadFM requires GPU — will OOM on CPU
assert torch.cuda.is_available(), "RadFM requires a GPU with ≥40GB VRAM"
device = "cuda"

# Load RadFM from the repo's multimodality_model.py
try:
    from src.model.multimodality_model import MultiLLaMAForCausalLM
    print("Imported MultiLLaMAForCausalLM from src.model")
except ImportError:
    from multimodality_model import MultiLLaMAForCausalLM
    print("Imported MultiLLaMAForCausalLM from local path")

# Load model with checkpoint
model = MultiLLaMAForCausalLM.from_pretrained(
    "/tmp/RadFM/Model/RadFM",
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()
print("RadFM loaded.")
print("Vision model type:", type(model.vision_encoder).__name__)

## 3. Image preprocessing

RadFM treats 2D images as 3D by repeating along depth (D=4).
Input format: `(B, C=3, H, W, D)` — a pseudo-3D volume.

In [ ]:
from torchvision import transforms
from torchvision.transforms.functional import InterpolationMode
from PIL import Image

INPUT_H, INPUT_W = 512, 512   # RadFM default spatial resolution
DEPTH = 4                      # 2D CXR repeated along depth dimension

def make_transform():
    return transforms.Compose([
        transforms.Resize((INPUT_H, INPUT_W), interpolation=InterpolationMode.BICUBIC),
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x.repeat(3, 1, 1)),   # 1ch → 3ch (C, H, W)
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std =[0.229, 0.224, 0.225],
        ),
    ])

transform = make_transform()

def preprocess_2d_as_3d(path: str) -> torch.Tensor:
    """Return [1, 3, H, W, D=4] pseudo-3D tensor for RadFM."""
    img = Image.open(path)
    x = transform(img)              # [3, H, W]
    x = x.unsqueeze(-1)             # [3, H, W, 1]
    x = x.repeat(1, 1, 1, DEPTH)   # [3, H, W, 4]
    return x.unsqueeze(0)           # [1, 3, H, W, 4]

dummy = torch.zeros(1, 3, INPUT_H, INPUT_W, DEPTH)
print("dummy input (3D):", dummy.shape)   # [1, 3, 512, 512, 4]

## 4. Extract visual embeddings (vision encoder only)

RadFM's vision pipeline: 3D ViT → Perceiver resampler → 32 fixed tokens per image.
For a global embedding, mean-pool the 32 Perceiver tokens.

In [ ]:
dummy_gpu = dummy.to(device, dtype=torch.float16)

with torch.no_grad():
    # Access the embedding layer directly (ViT + Perceiver)
    # RadFM's my_embedding_layer processes visual tokens
    try:
        # Try the embedding layer's vision processing directly
        vis_tokens = model.model.embed_tokens.vit_model(dummy_gpu)   # [B, 32, d]
    except AttributeError:
        # Fallback: use the full embedding module with a dummy text input
        # The vision encoder is typically at model.vision_encoder
        vis_tokens = model.vision_encoder(dummy_gpu)   # [B, 32, d]

print("Perceiver tokens:", vis_tokens.shape)         # [1, 32, d]
global_emb = vis_tokens.mean(dim=1)                  # [1, d]  — mean pool
print("Global embedding:", global_emb.shape)

## Segmentation mode — not applicable (Perceiver tokens)

RadFM compresses image features through a **Perceiver resampler** into 32 fixed
learnable query tokens. These tokens have no fixed spatial correspondence to image
regions — the Perceiver cross-attends globally.

A spatially regular `[B, D, H, W]` feature map is **not available** from the Perceiver
output. For dense prediction tasks, use a backbone with a standard ViT patch grid
(e.g. RAD-DINO 37×37, MedSigLIP 28×28, XraySigLIP 32×32).

In [ ]:
# RadFM uses a Perceiver resampler that compresses the image into 32 fixed query tokens.
# These tokens have NO spatial correspondence — the Perceiver cross-attends globally.
# A spatially regular [B, D, H, W] feature map is NOT available from the Perceiver output.

print("Perceiver tokens (not a spatial grid):", vis_tokens.shape)   # [B, 32, d]
print()
print("RadFM cannot produce a standard patch-grid feature map for dense segmentation.")
print("The 32 Perceiver output tokens are semantically rich but spatially unordered.")
print()
print("For dense prediction, prefer a ViT backbone with a regular patch grid:")
print("  RAD-DINO (37×37), MedSigLIP (28×28), XraySigLIP (32×32), BioViL-T (30×30)")

## 5. Alternative — use RadFM's embedding layer directly

In [ ]:
# RadFM stores the vision+Perceiver pipeline in my_embedding_layer.py
# Check where it lives in the loaded model
print("Model modules:")
for name, _ in list(model.named_children())[:10]:
    print(" ", name)

# Try to locate the visual embedding path
# Common locations in RadFM:
for attr in ["vision_encoder", "embed_tokens", "model"]:
    obj = getattr(model, attr, None)
    if obj is not None:
        print(f"Found model.{attr}:", type(obj).__name__)

## Notes for RadHarmony integration

```
model_call : lambda m, x: m.vision_encoder(x).mean(dim=1)   # [B, 32, d] → [B, d]
pool       : None  (already pooled in model_call)
embed_dim  : d  (check global_emb.shape[-1] above — typically 4096 or 5120)
transform  : Resize(512,512,BICUBIC) → Grayscale → ToTensor → repeat(3ch)
             → Normalize(ImageNet) → repeat depth dim (×4) → [3, H, W, 4]
```

**Practical considerations**:
- RadFM is a 26GB model (LLaMA-7B + vision encoder). For pure classification
  embedding tasks, the visual tokens (32 per image) are a very high-dimensional
  representation — expensive to store for large datasets.
- If the goal is only embedding extraction (not VQA), **a lighter backbone
  (RAD-DINO, BioViL-T, CheXzero) will be far more practical** for linear probing.
- The `model_call` path above needs verification against the actual module structure
  (cell 5's diagnostic output) — RadFM's codebase has evolved across versions.

**Hardware**: A100 80GB recommended. With `torch_dtype=float16` and `device_map='auto'`,
it runs on 40GB but leaves little headroom for batch inference.

**Segmentation mode**: Not applicable — RadFM's Perceiver resampler outputs 32 spatially
unordered tokens. Use a ViT backbone with a standard patch grid for dense prediction.

In [ ]:
# Full generation example (from RadFM repo)
# This requires the tokenizer and specific input format

CXR_PATH = "/path/to/cxr.jpg"
QUESTION = "What are the findings in this chest X-ray?"

try:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained("/tmp/RadFM/Model/RadFM")

    # Prepare image
    img_3d = preprocess_2d_as_3d(CXR_PATH).to(device, dtype=torch.float16)

    # Tokenize question with image placeholder
    # RadFM uses <image> tokens — exact format depends on tokenizer config
    # See RadFM repo's inference script for the exact template
    inputs = tokenizer(QUESTION, return_tensors="pt").to(device)

    with torch.no_grad():
        generated = model.generate(
            **inputs,
            images=img_3d,
            max_new_tokens=200,
        )

    print(tokenizer.decode(generated[0], skip_special_tokens=True))
except Exception as e:
    print(f"Full generation requires matching tokenizer format: {e}")
    print("See /tmp/RadFM/src/generate.py for the correct input template.")

## Notes for RadHarmony integration

```
model_call : lambda m, x: m.vision_encoder(x).mean(dim=1)   # [B, 32, d] → [B, d]
pool       : None  (already pooled in model_call)
embed_dim  : d  (check global_emb.shape[-1] above — typically 4096 or 5120)
transform  : Resize(512,512,BICUBIC) → Grayscale → ToTensor → repeat(3ch)
             → Normalize(ImageNet) → repeat depth dim (×4) → [3, H, W, 4]
```

**Practical considerations**:
- RadFM is a 26GB model (LLaMA-7B + vision encoder). For pure classification
  embedding tasks, the visual tokens (32 per image) are a very high-dimensional
  representation — expensive to store for large datasets.
- If the goal is only embedding extraction (not VQA), **a lighter backbone
  (RAD-DINO, BioViL-T, CheXzero) will be far more practical** for linear probing.
- The `model_call` path above needs verification against the actual module structure
  (cell 5's diagnostic output) — RadFM's codebase has evolved across versions.

**Hardware**: A100 80GB recommended. With `torch_dtype=float16` and `device_map='auto'`,
it runs on 40GB but leaves little headroom for batch inference.